# What machine learning is, and how we will work

**Lecture 1** · Géron, Chapters 1–2

Applications of Machine Learning — BSc Mathematics of Artificial Intelligence

---

**How to use this notebook.** You are not expected to type the code. You are
expected to *read* it before you run it, and to be able to say what every line
does and what would break if it changed.

Every code cell is preceded by the **specification that would produce it** —
input, output, constraint, check. Read the box, work out what the check should
say, *then* run the cell. That order is the whole point of the box.

Run the cells in order. Anything that takes more than a few seconds says so,
and anything that needs a GPU says that too. Nothing here is wrong on purpose.

**About the prompt boxes.** Every code cell in this notebook is preceded by a
quoted prompt naming four things: the input, the output, the constraint the
method must respect, and a check whose answer you can work out before running
anything. Read the box, answer the check in your head, then run the cell.

The prompts are **specifications, not transcripts** — this is what you would
have to ask for in order to get this cell, not a recording of somebody asking
for it. If your own prompt is vaguer than the box, expect worse code than the
cell below it.


This notebook is the whole of Lecture 1's second half, in runnable form: the
brief, the data, the split, and the exploration. Nothing is fitted here — the
first model arrives in Lecture 2, on purpose. Looking properly at data before
modelling it is not a preliminary, it is the part that decides whether the
model can work at all.

Runs on free CPU in about two minutes.

## 1 · Setup

> **Prompt · setup**
>
> **input** · nothing
>
> **output** · the version of every library this notebook depends on, and one seed
>
> **constraint** · ASSERT the scikit-learn version rather than printing it — `root_mean_squared_error` arrived in 1.4, and on an older Colab image the failure is an ImportError twenty cells from here
>
> **check** · RANDOM_STATE is defined here ONCE and used for every split, every model and every shuffle in the notebook. A notebook carrying three different seeds cannot be reproduced by reading it

In [ ]:
# --- setup -------------------------------------------------------------------
# Not examinable: this is engineering hygiene, not machine learning. It is here
# because a version mismatch produces a confusing error twenty cells later.
import sys, sklearn, numpy as np, pandas as pd, matplotlib

print(f"python       {sys.version.split()[0]}")
print(f"scikit-learn {sklearn.__version__}")
print(f"numpy        {np.__version__}")
print(f"pandas       {pd.__version__}")

# root_mean_squared_error arrived in scikit-learn 1.4
assert tuple(int(p) for p in sklearn.__version__.split(".")[:2]) >= (1, 4), \
    "This notebook needs scikit-learn >= 1.4.  In Colab: %pip install -U scikit-learn"

RANDOM_STATE = 42          # every split, every model, every shuffle
pd.set_option("display.width", 100)

## 2 · The data

> **Prompt · the data**
>
> **input** · the California housing tarball
>
> **output** · 20,640 districts and 10 columns
>
> **constraint** · a FUNCTION that downloads if absent and reads if present — the data will change, and you will need this on another machine
>
> **check** · assert the shape, rather than trusting the download
>
> ---
>
> **try** · delete the `datasets/` directory and re-run the cell. If it cannot rebuild its own input from nothing, it is not reproducible — it is cached.

In [ ]:
# --- the data ----------------------------------------------------------------
# A function, not a manual download: the data will change, and you will need
# this on another machine.  ~5 s the first time, instant afterwards.
from pathlib import Path
import tarfile, urllib.request

def load_housing():
    tarball = Path("datasets/housing.tgz")
    if not tarball.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball)
        with tarfile.open(tarball) as t:
            t.extractall(path="datasets", filter="data")
    return pd.read_csv("datasets/housing/housing.csv")

housing_full = load_housing()

assert housing_full.shape == (20640, 10), f"unexpected shape {housing_full.shape}"
print(f"{len(housing_full):,} districts, {housing_full.shape[1]} columns")
housing_full.head()

### What is in it

Ten attributes per district. One of them is not numeric, and one column has
holes in it. Find both before reading on.

> **Prompt · what is in it**
>
> **input** · the loaded frame
>
> **output** · every column, its type and its non-null count
>
> **constraint** · `.info()`, not `.head()` — the two things worth finding here are a non-numeric column and a column with holes in it, and neither is visible in five rows
>
> **check** · the non-null count of nine columns equals the row count, and of one column it does not
>
> ---
>
> **try** · `housing_full.head()` instead. Which of the two findings above is still visible in five rows, and which is not?

In [ ]:
housing_full.info()

> **Prompt · count the holes, and the categories**
>
> **input** · the frame
>
> **output** · how many districts are missing total_bedrooms, and the counts of every category level
>
> **constraint** · print the missing count as a PERCENTAGE as well as a count — 207 sounds like a lot and 1% does not
>
> **check** · `value_counts()` on the categorical sums to 20,640, and one of its levels has n < 10
>
> ---
>
> **try** · `normalize=True` on the value_counts. ISLAND becomes 0.0002 — at what point does a rare level stop being a curiosity and start being a problem?

In [ ]:
n_missing = housing_full["total_bedrooms"].isna().sum()
print(f"total_bedrooms is missing in {n_missing} districts "
      f"({100 * n_missing / len(housing_full):.1f}%)")
print()
print(housing_full["ocean_proximity"].value_counts())

`ISLAND` has five districts in the whole of California. Remember that: a level
with n=5 is a level that will be absent from some cross-validation folds, which
matters from the next lecture onwards.

## 3 · A quick look at the whole set

One look at everything, to find what is *structurally* wrong with the data —
the kind of fact you need before you can decide anything at all. Then we split,
and from that point on every number is computed on the training half.

Two things should jump out of the histograms. Take thirty seconds before you
scroll past them.

> **Prompt · histograms**
>
> **input** · the full frame
>
> **output** · a histogram of every numeric column
>
> **constraint** · 50 bins, not the default 10 — a cap at the top of a distribution is one bar, and at 10 bins it is inside a bar with everything else
>
> **check** · nine panels, one per numeric column, and two of them have a conspicuous spike at their right-hand edge
>
> ---
>
> **try** · `bins=10`, the default. Both spikes vanish into a neighbouring bar. That is why the constraint is there.

In [ ]:
import matplotlib.pyplot as plt

housing_full.hist(bins=50, figsize=(12, 8))
plt.tight_layout(); plt.show()

**The income is not in dollars** — it is scaled, and capped at 15.0001.

**The target is capped too**, and the target is our label — so those districts
carry a label that is not the answer, and no model can be right about them.

That is enough to know before splitting. We *count* it, and look at the stripes
under it, after the split — on the training half, like every other number in
this notebook.

## 4 · Split before you explore

Everything you learn from the data *before* the split leaks into the choices you
make afterwards — through you, not through the code. There is no library that
prevents this, which is why it is a rule about the order of your own actions.

We stratify on income because the domain experts said income predicts price, and
because `median_income` is continuous, we band it first. Five bands, chosen so
that no band is tiny: stratification needs enough districts per stratum to be
worth doing.

> **Prompt · the stratified split**
>
> **input** · the whole frame
>
> **output** · an 80/20 split stratified on the income band
>
> **constraint** · stratify on the income BAND, not on the raw income — `train_test_split` stratifies on a categorical, and 20,640 distinct incomes are 20,640 strata
>
> **check** · the two halves sum to 20,640 and their indices are disjoint
>
> ---
>
> **try** · `stratify=housing_full["median_income"]`, the raw income, instead of the band. Read the error — it says exactly why the banding step exists.

In [ ]:
from sklearn.model_selection import train_test_split

# five bands, on the scaled income. The last is open-ended because the top of
# the distribution is thin and a fixed upper edge would leave a near-empty band.
income_cat = pd.cut(housing_full["median_income"],
                    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
                    labels=[1, 2, 3, 4, 5])

train_set, test_set = train_test_split(
    housing_full, test_size=0.2, random_state=RANDOM_STATE, stratify=income_cat)

# assert, do not hope
assert len(train_set) + len(test_set) == len(housing_full)
assert set(train_set.index).isdisjoint(test_set.index), "the split overlaps"
print(f"train {len(train_set):,}   test {len(test_set):,}")

# From here to the last cell of the NEXT lecture, `test_set` is not touched.
housing = train_set.copy()

### Did stratifying actually buy anything?

The claim is that a random split gets the income mix wrong and a stratified one
does not. That is a measurable claim, so measure it: take the proportion of
districts in each income band in the full dataset, and compare it with the
proportion each kind of split produces.

> **Prompt · sampling bias, measured**
>
> **input** · the income bands, the full frame, and one test set of each kind
>
> **output** · the band proportions under each, and the percentage error of each against the full-data proportions
>
> **constraint** · the same seed for both splits, so the only difference between them is the stratification
>
> **check** · the stratified error is smaller in every band; the interesting question is by how much
>
> ---
>
> **try** · change `random_state` to 0, then 1, then 2. The random error moves every time; does the stratified one?

In [ ]:
random_test = train_test_split(housing_full, test_size=0.2,
                               random_state=RANDOM_STATE)[1]

def band_share(frame):
    return income_cat.loc[frame.index].value_counts(normalize=True).sort_index()

overall = income_cat.value_counts(normalize=True).sort_index()
comparison = pd.DataFrame({
    "overall %":    100 * overall,
    "stratified %": 100 * band_share(test_set),
    "random %":     100 * band_share(random_test),
})
comparison["stratified error %"] = (
    100 * (comparison["stratified %"] / comparison["overall %"] - 1))
comparison["random error %"] = (
    100 * (comparison["random %"] / comparison["overall %"] - 1))

print(comparison.round(2).to_string())
print(f"\nworst error — stratified {comparison['stratified error %'].abs().max():.2f}%"
      f"   random {comparison['random error %'].abs().max():.2f}%")

## 5 · Now explore — the training set, and only that

Everything below is computed on `housing`, the training copy. Every correlation,
every scatter, every ratio. The 4,128 test districts play no part in any decision
made from here on.

### First, the cap — counted, not squinted at

> **Prompt · count the cap**
>
> **input** · the training half's target column
>
> **output** · how many training districts sit at the cap, and the commonest values below it
>
> **constraint** · count it — a histogram shows you a spike, and a count tells you whether it is 5% of your labels or 0.5%
>
> **check** · the commonest values below the cap are all multiples of the same number; work out which before running it
>
> ---
>
> **try** · raise the threshold from 500,000 to 500,001. The count does not change — what does that tell you about how the cap was applied?

In [ ]:
capped = (housing["median_house_value"] >= 500_000).sum()
print(f"{capped} districts sit at the cap "
      f"({100 * capped / len(housing):.1f}% of the training set)")

# a continuous target should have an almost flat value_counts. Where it is not,
# the recording process is visible — a fact about the survey, not California.
counts = housing["median_house_value"].value_counts()
print(f"\na typical price is shared by {counts.median():.0f} districts")
print("\nthe five commonest values below the cap:")
print(counts.drop(counts.index.max()).head(5))

Every one of those is a multiple of **$12,500** — artefacts of how the survey
recorded prices, not facts about California.

The cap is the one that matters. The target is the *label*, so a capped district
has a label that is not the answer and no model can be right about it. Two
responses are legitimate, and the choice belongs to the stakeholder rather than
to us: collect proper labels for those districts, or drop them from both halves
and state that the system does not predict above $500,000.

A well-known description of this dataset also names fainter lines at \$450,000,
\$350,000 and \$280,000. When a source names specific numbers about your data,
those numbers are checkable.

> **Prompt · check the famous claim**
>
> **input** · three values named in a well-known description of this dataset
>
> **output** · how many training districts sit at each
>
> **constraint** · check the claim against the counts you just computed rather than repeating it
>
> **check** · compare each against the median count printed above — one of the three is real, one is marginal, and one is indistinguishable from the background
>
> ---
>
> **try** · three values nobody claimed — 460,000, 340,000 and 270,000. If those come back comparable, the original claim was about the background, not about the data.

In [ ]:
for value in (450_000, 350_000, 280_000):
    print(f"${value:>9,}  {counts.get(value, 0):>4d} districts")

### Then the geography

> **Prompt · geography**
>
> **input** · the training districts' longitude and latitude
>
> **output** · a scatter of California, with population as the marker size and price as the colour
>
> **constraint** · alpha well below 1 — at alpha=1 the dense areas saturate into a solid blob and the density information, which is the point of the plot, is destroyed
>
> **check** · the coastline is legible, and the expensive districts are visibly not uniformly distributed
>
> ---
>
> **try** · `alpha=1`, then separately `cmap="jet"`. Two different lessons, and the second is easier to see than to explain.

In [ ]:
housing.plot(kind="scatter", x="longitude", y="latitude",
             alpha=0.2,                       # density, not just position
             s=housing["population"] / 100, label="population",
             c="median_house_value", cmap="viridis",   # not jet: see below
             colorbar=True, figsize=(9, 6), sharex=False)
plt.title("training districts: size = population, colour = median price")
plt.tight_layout(); plt.show()

The colormap is `viridis` rather than the `jet` you will see in older code.
`jet` is not perceptually uniform: it has a bright band in the middle and dark
ends, so equal steps in the data are not equal steps in apparent brightness, and
it invents boundaries in smooth data that a reader then interprets as structure.
It also collapses to an unreadable grey ramp when printed or seen by a
colour-blind reader. `viridis` is monotone in lightness and survives both.

Price is high near the ocean and near the two big cities. That is a fact you can
use — and one we will make explicit as a feature in the next lecture.

> **Prompt · correlations**
>
> **input** · the numeric training columns
>
> **output** · the linear correlation of every attribute with the target, ranked
>
> **constraint** · Pearson only, and say so — it measures LINEAR association and nothing else
>
> **check** · median_income is far the strongest; every other column is below 0.15 in absolute value
>
> ---
>
> **try** · `method="spearman"`, which ranks rather than measures. Which column moves most, and what does its histogram above look like?

In [ ]:
corr = housing.select_dtypes(include=[np.number]).corr(numeric_only=True)
print("linear (Pearson) correlation with the target:\n")
print(corr["median_house_value"].sort_values(ascending=False).round(3).to_string())

`median_income` at about 0.69 is far and away the strongest single predictor,
which is why we stratified on it. But read the weak entries carefully rather
than dismissing them: `total_rooms` correlates with the target at about 0.14,
and that is not because the number of rooms is irrelevant to price. It is
because `total_rooms` is a *district* total, so it mostly measures how many
people live in the district.

The quantity that should matter is rooms **per household**. Correlation cannot
tell you that; only knowing what the column means can.

> **Prompt · attribute combinations**
>
> **input** · the training frame
>
> **output** · three per-household and per-room ratios, and their correlation with the target
>
> **constraint** · ratios, not totals — a district total is a proxy for district size, and district size is not what we are predicting
>
> **check** · at least one ratio correlates more strongly than either column it was built from
>
> ---
>
> **try** · add `bedrooms_per_person`. It is a ratio too — is it any use? Not every combination is worth having, and the correlation is how you find out.

In [ ]:
housing["rooms_per_house"]   = housing["total_rooms"] / housing["households"]
housing["bedrooms_ratio"]    = housing["total_bedrooms"] / housing["total_rooms"]
housing["people_per_house"]  = housing["population"] / housing["households"]

new_corr = housing.select_dtypes(include=[np.number]).corr(
    numeric_only=True)["median_house_value"]

for name in ("rooms_per_house", "bedrooms_ratio", "people_per_house",
             "total_rooms", "population", "households"):
    print(f"{name:20s} {new_corr[name]:+.3f}")

`bedrooms_ratio` is more strongly correlated with price than any of the three
raw columns it was derived from — a district where a small share of the rooms
are bedrooms is a district of larger, more expensive houses. Nothing in the data
told us to compute that. Knowing what the columns *mean* did.

One warning to carry into the next lecture: when you build combined features,
avoid simple weighted sums of columns you already have. A feature that is a
linear combination of existing ones adds no information and makes the linear
algebra worse — the next lecture derives exactly why, when it derives the normal
equation.

## 6 · Where we are

- The data has one categorical column, 207 missing values in `total_bedrooms`,
  a target capped at $500,000, and per-district totals that mostly measure
  district size.
- The split is done, stratified on income, and the test set is sealed.
- The strongest single predictor is `median_income`; the most useful engineered
  feature is `bedrooms_ratio`.

Nothing has been fitted. **Next lecture:** the normal equation, then a
preprocessing pipeline that handles all of the above, cross-validation, and the
first honest number.

**Before then:** run this notebook top to bottom once. Then change the income
bands in `pd.cut` — try three bands, or eight — and re-run from that cell to the
sampling-bias table. What happens to the stratified error, and why?